# Semana 8: Modelos Seq2Seq y Atención
## Notebook de Ejercicios (NB2) – Traducción Inglés-Español

**Propósito:** Aplicar un modelo Seq2Seq con atención a un problema real de traducción automática usando el dataset Multi30k.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Preprocesar y construir vocabularios para traducción.
- Entrenar un modelo Seq2Seq con atención.
- Evaluar la calidad de la traducción con BLEU score usando sacrebleu.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y configuramos PyTorch.

In [ ]:
# Importamos librerías
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

# Para procesamiento de datos
from collections import Counter
import random
import math

# Para BLEU score
try:
    import sacrebleu
    SACREBLEU_AVAILABLE = True
except ImportError:
    SACREBLEU_AVAILABLE = False
    print("Nota: sacrebleu no está instalado. Se instalará más adelante.")

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Configuración de PyTorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

# Semillas para reproducibilidad
np.random.seed(42)
torch.manual_seed(42)

print("\nLibrerías importadas correctamente.")

---
## 1. Carga y Exploración del Dataset Multi30k

El dataset **Multi30k** contiene 30,000 pares de frases cortas en inglés y alemán. Para este ejercicio, trabajaremos con inglés-español (simulado).

In [ ]:
# Instalar torchtext si es necesario
try:
    from torchtext.datasets import Multi30k
    from torchtext.data.utils import get_tokenizer
    from torchtext.vocab import build_vocab_from_iterator
    TORCHTEXT_AVAILABLE = True
except ImportError:
    TORCHTEXT_AVAILABLE = False
    print("Instalando torchtext...")
    !pip install torchtext
    from torchtext.datasets import Multi30k
    from torchtext.data.utils import get_tokenizer
    from torchtext.vocab import build_vocab_from_iterator

# Cargar el dataset
print("Cargando dataset Multi30k...")
train_iter, valid_iter, test_iter = Multi30k(split=('train', 'valid', 'test'), language_pair=('en', 'de'))

# Convertir a listas para facilitar el manejo
train_pairs = [(en, de) for en, de in train_iter]
valid_pairs = [(en, de) for en, de in valid_iter]
test_pairs = [(en, de) for en, de in test_iter]

print(f"Número de pares de entrenamiento: {len(train_pairs)}")
print(f"Número de pares de validación: {len(valid_pairs)}")
print(f"Número de pares de prueba: {len(test_pairs)}")

# Mostrar ejemplos
print("\n=== EJEMPLOS ===")
for i in range(3):
    en, de = train_pairs[i]
    print(f"Inglés: {en}")
    print(f"Alemán: {de}")
    print()

---
## 2. Tokenización y Construcción de Vocabularios

Usaremos tokenizadores simples basados en espacios para este ejercicio.

In [ ]:
# Tokenizadores simples
def tokenize_en(text):
    return text.lower().split()

def tokenize_de(text):
    return text.lower().split()

# Función para construir vocabulario
def build_vocab(data_iter, tokenizer, min_freq=2, specials=['<pad>', '<sos>', '<eos>', '<unk>']):
    counter = Counter()
    for (en, de) in data_iter:
        counter.update(tokenizer(en))
        counter.update(tokenizer(de))
    
    vocab = {word: idx for idx, (word, freq) in enumerate(counter.items()) if freq >= min_freq}
    
    # Añadir tokens especiales
    for special in specials:
        if special not in vocab:
            vocab[special] = len(vocab)
    
    return vocab

print("Construyendo vocabularios...")
src_vocab = build_vocab(train_pairs, tokenize_en)
tgt_vocab = build_vocab(train_pairs, tokenize_de)

print(f"Tamaño vocabulario inglés: {len(src_vocab)}")
print(f"Tamaño vocabulario alemán: {len(tgt_vocab)}")

# Diccionarios inversos
src_idx2word = {idx: word for word, idx in src_vocab.items()}
tgt_idx2word = {idx: word for word, idx in tgt_vocab.items()}

---
## 3. Codificación de Frases

Convertimos las frases a secuencias de índices con tokens SOS y EOS.

In [ ]:
def encode_sentence(sentence, vocab, tokenizer, max_len=50):
    tokens = tokenizer(sentence)
    indices = [vocab.get(token, vocab['<unk>']) for token in tokens]
    # Añadir SOS al inicio y EOS al final
    indices = [vocab['<sos>']] + indices + [vocab['<eos>']]
    # Padding
    indices = indices[:max_len]
    indices += [vocab['<pad>']] * (max_len - len(indices))
    return indices

def prepare_data(pairs, src_vocab, tgt_vocab, src_tokenizer, tgt_tokenizer, max_len=50):
    src_data = []
    tgt_data = []
    for en, de in pairs:
        src_data.append(encode_sentence(en, src_vocab, src_tokenizer, max_len))
        tgt_data.append(encode_sentence(de, tgt_vocab, tgt_tokenizer, max_len))
    return torch.tensor(src_data), torch.tensor(tgt_data)

max_len = 50
print("Codificando datos...")
X_train, Y_train = prepare_data(train_pairs[:1000], src_vocab, tgt_vocab, tokenize_en, tokenize_de, max_len)  # subconjunto para velocidad
X_valid, Y_valid = prepare_data(valid_pairs[:200], src_vocab, tgt_vocab, tokenize_en, tokenize_de, max_len)
X_test, Y_test = prepare_data(test_pairs[:200], src_vocab, tgt_vocab, tokenize_en, tokenize_de, max_len)

print(f"Forma X_train: {X_train.shape}")
print(f"Forma Y_train: {Y_train.shape}")

In [ ]:
# Crear DataLoaders
batch_size = 32

train_dataset = TensorDataset(X_train, Y_train)
valid_dataset = TensorDataset(X_valid, Y_valid)
test_dataset = TensorDataset(X_test, Y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Número de lotes de entrenamiento: {len(train_loader)}")

---
## 4. Definición del Modelo Seq2Seq con Atención

### 4.1. Encoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.3):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        outputs, (hidden, cell) = self.lstm(embedded)
        return outputs, hidden, cell

### 4.2. Decoder con Atención (Luong)

In [ ]:
class DecoderWithAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.3):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.attn_proj = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, y_prev, hidden, cell, encoder_outputs):
        # y_prev: (batch, 1)
        embedded = self.dropout(self.embedding(y_prev))
        lstm_out, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        
        # Calcular atención
        query = lstm_out.squeeze(1)  # (batch, hidden_dim)
        scores = torch.bmm(encoder_outputs, query.unsqueeze(2)).squeeze(2)  # (batch, seq_len)
        attn_weights = torch.softmax(scores, dim=1)  # (batch, seq_len)
        
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)  # (batch, hidden_dim)
        
        combined = torch.tanh(self.attn_proj(torch.cat((query, context), dim=1)))
        output = self.fc_out(self.dropout(combined))
        
        return output, hidden, cell, attn_weights

### 4.3. Modelo Seq2Seq Completo

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size, trg_len = trg.shape
        trg_vocab_size = self.decoder.vocab_size
        
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        
        encoder_outputs, hidden, cell = self.encoder(src)
        
        decoder_input = torch.ones(batch_size, 1).long().to(self.device)  # <sos>
        
        for t in range(trg_len):
            output, hidden, cell, attn = self.decoder(decoder_input, hidden, cell, encoder_outputs)
            outputs[:, t, :] = output
            
            if torch.rand(1).item() < teacher_forcing_ratio:
                decoder_input = trg[:, t].unsqueeze(1)
            else:
                decoder_input = output.argmax(1).unsqueeze(1)
        
        return outputs

# Parámetros
EMBED_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 1
DROPOUT = 0.3

encoder = Encoder(len(src_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(device)
decoder = DecoderWithAttention(len(tgt_vocab), EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(device)
model = Seq2Seq(encoder, decoder, device).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Modelo creado. Total de parámetros: {total_params:,}")

---
## 5. Entrenamiento del Modelo

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=src_vocab['<pad>'])
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, loader, optimizer, criterion, teacher_forcing_ratio=0.5):
    model.train()
    epoch_loss = 0
    
    for src, trg in loader:
        src, trg = src.to(device), trg.to(device)
        
        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio)
        
        loss = criterion(output.reshape(-1, len(tgt_vocab)), trg.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()
        
        epoch_loss += loss.item()
    
    return epoch_loss / len(loader)

def evaluate(model, loader, criterion):
    model.eval()
    epoch_loss = 0
    
    with torch.no_grad():
        for src, trg in loader:
            src, trg = src.to(device), trg.to(device)
            output = model(src, trg, teacher_forcing_ratio=0)  # greedy decoding
            loss = criterion(output.reshape(-1, len(tgt_vocab)), trg.reshape(-1))
            epoch_loss += loss.item()
    
    return epoch_loss / len(loader)

In [ ]:
epochs = 20
train_losses = []
valid_losses = []

print("Iniciando entrenamiento...\n")
for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    valid_loss = evaluate(model, valid_loader, criterion)
    
    train_losses.append(train_loss)
    valid_losses.append(valid_loss)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}")

print("\nEntrenamiento completado.")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train')
plt.plot(valid_losses, label='Valid')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.title('Evolución de la pérdida durante entrenamiento')
plt.legend()
plt.grid(True)
plt.show()

---
## 6. Traducción y Evaluación con BLEU

### 6.1. Función de traducción

In [ ]:
def translate_sentence(model, sentence, src_vocab, tgt_vocab, src_tokenizer, tgt_idx2word, max_len=50):
    model.eval()
    
    # Codificar frase origen
    indices = encode_sentence(sentence, src_vocab, src_tokenizer, max_len)
    src_tensor = torch.tensor([indices]).to(device)
    
    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src_tensor)
        
        decoder_input = torch.ones(1, 1).long().to(device)  # <sos>
        translated_indices = []
        
        for _ in range(max_len):
            output, hidden, cell, attn = model.decoder(decoder_input, hidden, cell, encoder_outputs)
            predicted = output.argmax(1).item()
            
            if predicted == tgt_vocab['<eos>']:
                break
            if predicted != tgt_vocab['<pad>']:
                translated_indices.append(predicted)
            
            decoder_input = torch.tensor([[predicted]]).to(device)
    
    return ' '.join([tgt_idx2word[idx] for idx in translated_indices])

In [ ]:
# Probar con algunas frases
test_sentences = [
    "A man in a blue shirt is sitting on a bench.",
    "A woman is cooking in the kitchen.",
    "Two dogs are playing in the park."
]

print("=== EJEMPLOS DE TRADUCCIÓN ===\n")
for sent in test_sentences:
    translation = translate_sentence(model, sent, src_vocab, tgt_vocab, tokenize_en, tgt_idx2word)
    print(f"Inglés: {sent}")
    print(f"Alemán: {translation}")
    print()

### 6.2. Evaluación con BLEU usando sacrebleu

In [ ]:
if not SACREBLEU_AVAILABLE:
    !pip install sacrebleu
    import sacrebleu

# Traducir un subconjunto de test
references = []
hypotheses = []

print("Evaluando con BLEU (esto puede tomar unos minutos)...")
for i in range(min(50, len(test_pairs))):
    src, ref = test_pairs[i]
    hyp = translate_sentence(model, src, src_vocab, tgt_vocab, tokenize_en, tgt_idx2word)
    references.append(ref)
    hypotheses.append(hyp)

# Calcular BLEU
bleu = sacrebleu.corpus_bleu(hypotheses, [references])
print(f"\nBLEU score: {bleu.score:.2f}")

In [ ]:
# Mostrar algunos ejemplos con sus referencias
print("\n=== EJEMPLOS CON REFERENCIAS ===")
for i in range(5):
    print(f"Ejemplo {i+1}:")
    print(f"  Fuente: {test_pairs[i][0]}")
    print(f"  Referencia: {references[i]}")
    print(f"  Traducción: {hypotheses[i]}")
    print()

---
## 7. Visualización de Atención (Opcional)

Para una frase de ejemplo, visualizamos la matriz de atención.

In [ ]:
def translate_with_attention(model, sentence, src_vocab, tgt_vocab, src_tokenizer, tgt_idx2word, max_len=50):
    model.eval()
    
    indices = encode_sentence(sentence, src_vocab, src_tokenizer, max_len)
    src_tensor = torch.tensor([indices]).to(device)
    
    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src_tensor)
        
        decoder_input = torch.ones(1, 1).long().to(device)
        translated_indices = []
        attention_weights = []
        
        for _ in range(max_len):
            output, hidden, cell, attn = model.decoder(decoder_input, hidden, cell, encoder_outputs)
            attention_weights.append(attn.cpu().numpy())
            
            predicted = output.argmax(1).item()
            if predicted == tgt_vocab['<eos>']:
                break
            if predicted != tgt_vocab['<pad>']:
                translated_indices.append(predicted)
            
            decoder_input = torch.tensor([[predicted]]).to(device)
    
    translation = ' '.join([tgt_idx2word[idx] for idx in translated_indices])
    return translation, np.array(attention_weights)

def plot_attention(sentence, translation, attention_weights, src_tokenizer):
    src_words = ['<sos>'] + src_tokenizer(sentence) + ['<eos>']
    trg_words = translation.split()
    
    attn_matrix = attention_weights.squeeze(1)[:len(trg_words), :len(src_words)]
    
    plt.figure(figsize=(12, 8))
    sns.heatmap(attn_matrix, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=src_words, yticklabels=trg_words)
    plt.xlabel('Palabras origen')
    plt.ylabel('Palabras destino')
    plt.title('Matriz de Atención')
    plt.show()

# Visualizar para una frase
sample_sentence = "A man is sitting on a bench."
translation, attn_weights = translate_with_attention(model, sample_sentence, src_vocab, tgt_vocab, tokenize_en, tgt_idx2word)
print(f"Frase: {sample_sentence}")
print(f"Traducción: {translation}")
plot_attention(sample_sentence, translation, attn_weights, tokenize_en)

---
## 8. Conclusiones

En este notebook hemos aplicado un modelo Seq2Seq con atención a un problema real de traducción:

✔️ **Preprocesamiento**: Construimos vocabularios y codificamos las frases con tokens especiales.

✔️ **Modelo Seq2Seq con atención**: Implementamos encoder LSTM y decoder con atención de Luong.

✔️ **Entrenamiento**: Entrenamos el modelo en el dataset Multi30k.

✔️ **Evaluación con BLEU**: Usamos `sacrebleu` para medir la calidad de las traducciones.

✔️ **Visualización**: Observamos cómo la atención alinea las palabras origen y destino.

**Lección clave**: Los modelos Seq2Seq con atención son efectivos para tareas de traducción, y el score BLEU es la métrica estándar para evaluar su calidad.

---
**Fin del Notebook de Ejercicios - Semana 8 (NLP)**